In [1]:
import pandas as pd
import os
from datetime import timedelta

In [ ]:


# Base directory (adjust if needed; based on your terminal path)
base_dir = '../'
hosp_dir = os.path.join(base_dir, 'physionet.org/files/mimiciv/3.1/hosp')
note_dir = os.path.join(base_dir, 'physionet.org/files/mimic-iv-note/2.2/note')
ext_dir = os.path.join(base_dir, 'physionet.org/files/mimic-iv-ext-22mcts/1.0.0')

# Key file paths for 30-day readmission task
files = {
    'admissions': os.path.join(hosp_dir, 'admissions.csv.gz'),
    'patients': os.path.join(hosp_dir, 'patients.csv.gz'),
    'diagnoses_icd': os.path.join(hosp_dir, 'diagnoses_icd.csv.gz'),
    'discharge': os.path.join(note_dir, 'discharge.csv.gz'),
    'events': os.path.join(ext_dir, 'clinical_event_timestamp.csv')  # Not gzipped
}

# Function to load, summarize, and check for readmission relevance
def load_and_check(filepath, nrows=1000, sample_frac=0.1, is_large=False):
    print(f"\n=== Loading {os.path.basename(filepath)} ===")
    if not os.path.exists(filepath):
        print(f"❌ File not found: {filepath}")
        return None
    
    try:
        kwargs = {'compression': 'gzip'} if filepath.endswith('.gz') else {}
        if is_large:
            df = pd.read_csv(filepath, nrows=nrows, **kwargs)
        else:
            df = pd.read_csv(filepath, **kwargs)
            if sample_frac and len(df) > 1000:
                df = df.sample(frac=sample_frac, random_state=42).reset_index(drop=True)
        
        print(f"✅ Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print("\nSample data:\n", df.head(3))
        print("\nData types:\n", df.dtypes)
        print("\nBasic stats:\n", df.describe(include='all').round(2))
        
        # Task-specific checks
        if 'admissions' in filepath:
            df['admittime'] = pd.to_datetime(df['admittime'])
            df['dischtime'] = pd.to_datetime(df['dischtime'])
            df = df.sort_values(['subject_id', 'admittime'])
            df['next_admittime'] = df.groupby('subject_id')['admittime'].shift(-1)
            df['days_to_next'] = (df['next_admittime'] - df['dischtime']).dt.days
            readmit_rate = (df['days_to_next'] <= 30).mean()
            print(f"\nReadmission preview (≤30 days): {readmit_rate:.2%} ({(df['days_to_next'] <= 30).sum()}/{len(df)})")
            print("Sample readmission gaps:\n", df['days_to_next'].dropna().head())
        elif 'patients' in filepath:
            print(f"\nAdult patients (anchor_age >=18): {(df['anchor_age'] >= 18).sum()}/{len(df)}")
        elif 'diagnoses_icd' in filepath:
            print(f"\nTop ICD codes: \n{df['icd_code'].value_counts().head()}")
        elif 'discharge' in filepath:
            print(f"\nAvg text length: {df['text'].str.len().mean():.0f} chars (sample: {df['text'].str[:100].iloc[0]}...)")
        elif 'events' in filepath:
            print(f"\nTop events: \n{df['Event'].value_counts().head()}")
            print(f"Timestamp range: {df['Time'].min()} to {df['Time'].max()} hours")
        
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Load key files for readmission task
admissions_df = load_and_check(files['admissions'], nrows=None)  # Full for label computation
patients_df = load_and_check(files['patients'])
diagnoses_df = load_and_check(files['diagnoses_icd'], nrows=5000)  # More rows for codes
discharge_df = load_and_check(files['discharge'], nrows=500)  # Text-heavy, small sample
events_df = load_and_check(files['events'], nrows=10000, is_large=True)  # 22M rows, heavy sample


SyntaxError: invalid syntax (1603769328.py, line 6)

In [ ]:
# Quick merge preview for cohort (small samples)
print("\n=== Cohort Preview (Admissions + Patients + Discharge) ===")
if all(df is not None for df in [admissions_df, patients_df, discharge_df]):
    # Sample for merge
    adm_samp = admissions_df.sample(1000, random_state=42)
    pat_samp = patients_df[['subject_id', 'gender', 'anchor_age']]
    dis_samp = discharge_df[['hadm_id', 'text']].sample(100, random_state=42)  # Fewer texts
    cohort = adm_samp.merge(pat_samp, on='subject_id', how='left')
    cohort = cohort.merge(dis_samp, on='hadm_id', how='left')
    print(f"Shape after merge: {cohort.shape}")
    print(cohort[['subject_id', 'hadm_id', 'gender', 'anchor_age', 'admission_type', 'text']].head(2))
    print("\nText availability: ", (cohort['text'].notna()).mean())

# Save small cohort for further EDA if needed
# cohort.to_csv('readmission_cohort_sample.csv', index=False)
print("\n✅ EDA complete! Use 'cohort' for plots (e.g., cohort['anchor_age'].hist())")